In [17]:
print("hi")

hi


In [27]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

df = pd.read_csv('spectral_feature_data.csv')
target_cols = [col for col in df.columns if col.startswith("p")]

# Select features: All columns that are NOT in the exclusion list
# This assumes your CSV contains: Spectral_Cols, ph, ec, Target_Cols, and ID
non_feature_cols = [col for col in df.columns if col.startswith('p4')]

feature_cols = [col for col in df.columns if col not in non_feature_cols]

#print(f"{target_cols}\n\n{non_feature_cols}\n\n{feature_cols}")
 
#input features
spectral_columns = [col for col in feature_cols if not col.startswith("p")]

'''prediction_columns.extend(["p1.pH.index", 'p1.EC.ds_m'])
X = df[prediction_columns]
print(X.columns)'''


'prediction_columns.extend(["p1.pH.index", \'p1.EC.ds_m\'])\nX = df[prediction_columns]\nprint(X.columns)'

In [37]:


results = []

# Assuming target_cols and base_spectral_columns are already defined
# Example: base_spectral_columns = ['410', '435', '460', '485', ...]

combinations = {"Spectral Only": [False, False],
                "Spectral + pH": [True, False],
                "Spectral + EC": [False, True],
                "Spectral + pH + EC": [True, True]}

imputer = SimpleImputer(strategy='mean')

for config_type in combinations.keys():
    print(f"\n========== Evaluating Config: {config_type} ==========")
    
    # FIX 1: Reset prediction columns to just the base spectral columns for this loop
    prediction_columns = spectral_columns.copy()
    
    maskpH = pd.Series(True, index=df.index)
    maskEC = pd.Series(True, index=df.index)

    if combinations[config_type][0]: # if we are to use ph value
        prediction_columns.append("p1.pH.index")
        maskpH = df["p1.pH.index"].notna()  # FIX 2: Check df, not X
        
    if combinations[config_type][1]: # if we are to use EC value
        prediction_columns.append("p1.EC.ds_m")
        maskEC = df["p1.EC.ds_m"].notna()   # FIX 2: Check df, not X

    # Update X for this specific configuration
    X = df[prediction_columns]
    
    # Combine the feature masks (rows where required features exist)
    feature_mask = maskpH & maskEC
    
    print(f"Features in use: {len(prediction_columns)} columns")
    
    for target in target_cols:
        
        if target not in df.columns:
            continue
            
        # FIX 3: Combine target mask with the feature masks properly
        target_mask = df[target].notna()
        final_mask = target_mask & feature_mask 
        
        y_clean = df.loc[final_mask, target]
        X_clean = X.loc[final_mask]
        
        if y_clean.shape[0] < 100:
            print(f"  -> Skipping {target}: Not enough data ({y_clean.shape[0]} rows).")
            continue
            
        # FIX 4: Handle NaNs in spectral columns to prevent PLSR crash
        X_clean_imputed = imputer.fit_transform(X_clean)

        X_train, X_test, y_train, y_test = train_test_split(X_clean_imputed, y_clean, test_size=0.2, random_state=42)

        # Setup Grid Search
        # FIX 5: Ensure max components doesn't exceed available features
        max_comp = min(20, X_train.shape[1])
        param_grid = {'n_components': np.arange(1, max_comp + 1)}
        
        pls = PLSRegression()
        grid_search = GridSearchCV(pls, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
        
        try:
            grid_search.fit(X_train, y_train)
        except Exception as e:
            print(f"  -> Error training {target}: {e}")
            continue
        
        best_pls = grid_search.best_estimator_
        
        y_pred = best_pls.predict(X_test).flatten()
        
        metrics = {
            'Config': config_type, # NEW: Track which config produced this score
            'Feature': target,
            'Best_Components': grid_search.best_params_['n_components'],
            'R2': r2_score(y_test, y_pred),
            'MAE': mean_absolute_error(y_test, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
            "MPE": np.mean((y_test - y_pred) / y_test) * 100
        }
        
        results.append(metrics)
        print(f"  -> Finished {target}: R2 = {metrics['R2']:.3f} | MAE = {metrics['MAE']:.3f}")

# Save and View Performance Table
if results:
    performance_df = pd.DataFrame(results)
    performance_df.to_csv('plsr_performance_results_combinations.csv', index=False)
    print("\n--- PLSR Performance Summary ---")
    print(performance_df)


========== Evaluating Config: Spectral Only ==========
Features in use: 18 columns
  -> Finished p1.pH.index: R2 = 0.532 | MAE = 0.749
  -> Finished p1.EC.ds_m: R2 = 0.132 | MAE = 0.172
  -> Finished p1.Clay.wt_pct: R2 = 0.329 | MAE = 14.712
  -> Finished p1.Sand.wt_pct: R2 = 0.133 | MAE = 9.043
  -> Finished p1.Silt.wt_pct: R2 = 0.255 | MAE = 14.260
  -> Finished p2.N.wt_pct: R2 = 0.467 | MAE = 0.177
  -> Skipping p2.Zn.mg_kg: Not enough data (0 rows).
  -> Finished p2.OC.wt_pct: R2 = 0.413 | MAE = 3.858
  -> Skipping p3.Fe.mg_kg: Not enough data (0 rows).


c:\Users\jagda\anaconda3\Lib\site-packages\numpy\_core\_methods.py:53: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


  -> Finished p3.K.mg_kg: R2 = 0.119 | MAE = 229.902


c:\Users\jagda\anaconda3\Lib\site-packages\numpy\_core\_methods.py:53: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


  -> Finished p3.P.mg_kg: R2 = 0.166 | MAE = 19.671
  -> Finished p3.S.wt_pct: R2 = 0.069 | MAE = 0.012
  -> Finished p4.BD.g_cm3: R2 = 0.254 | MAE = 0.202
  -> Finished p4.CEC.cmolc_kg: R2 = 0.219 | MAE = 8.180
  -> Finished p4.CF.wt_pct: R2 = 0.095 | MAE = 8.665
  -> Finished p4.WR_10kPa.wt_pct: R2 = 0.140 | MAE = 9.215
  -> Finished p4.WR_1500kPa.wt_pct: R2 = 0.148 | MAE = 8.055
  -> Finished p4.WR_33kPa.wt_pct: R2 = 0.235 | MAE = 9.672

========== Evaluating Config: Spectral + pH ==========
Features in use: 19 columns
  -> Finished p1.pH.index: R2 = 1.000 | MAE = 0.000
  -> Finished p1.EC.ds_m: R2 = 0.124 | MAE = 0.172
  -> Finished p1.Clay.wt_pct: R2 = 0.333 | MAE = 14.994
  -> Finished p1.Sand.wt_pct: R2 = 0.150 | MAE = 9.040
  -> Finished p1.Silt.wt_pct: R2 = 0.255 | MAE = 13.874
  -> Finished p2.N.wt_pct: R2 = 0.465 | MAE = 0.174
  -> Skipping p2.Zn.mg_kg: Not enough data (0 rows).
  -> Finished p2.OC.wt_pct: R2 = 0.482 | MAE = 3.643
  -> Skipping p3.Fe.mg_kg: Not enough data (

...

c:\Users\jagda\anaconda3\Lib\site-packages\numpy\_core\_methods.py:53: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


  -> Finished p3.P.mg_kg: R2 = 0.166 | MAE = 19.676
  -> Skipping p3.S.wt_pct: Not enough data (50 rows).
  -> Finished p4.BD.g_cm3: R2 = 0.312 | MAE = 0.168
  -> Finished p4.CEC.cmolc_kg: R2 = 0.292 | MAE = 7.665
  -> Finished p4.CF.wt_pct: R2 = 0.098 | MAE = 8.661
  -> Finished p4.WR_10kPa.wt_pct: R2 = 0.105 | MAE = 9.756
  -> Finished p4.WR_1500kPa.wt_pct: R2 = 0.193 | MAE = 7.997
  -> Finished p4.WR_33kPa.wt_pct: R2 = 0.134 | MAE = 9.582

========== Evaluating Config: Spectral + EC ==========
Features in use: 19 columns
  -> Finished p1.pH.index: R2 = 0.623 | MAE = 0.665
  -> Finished p1.EC.ds_m: R2 = 1.000 | MAE = 0.000
  -> Skipping p1.Clay.wt_pct: Not enough data (50 rows).
  -> Finished p1.Sand.wt_pct: R2 = 0.300 | MAE = 3.541
  -> Skipping p1.Silt.wt_pct: Not enough data (50 rows).
  -> Finished p2.N.wt_pct: R2 = 0.598 | MAE = 0.157
  -> Skipping p2.Zn.mg_kg: Not enough data (0 rows).
  -> Finished p2.OC.wt_pct: R2 = 0.511 | MAE = 3.449
  -> Skipping p3.Fe.mg_kg: Not enough da

c:\Users\jagda\anaconda3\Lib\site-packages\numpy\_core\_methods.py:53: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


  -> Finished p3.P.mg_kg: R2 = 0.217 | MAE = 18.252
  -> Skipping p3.S.wt_pct: Not enough data (51 rows).
  -> Skipping p4.BD.g_cm3: Not enough data (51 rows).
  -> Skipping p4.CEC.cmolc_kg: Not enough data (51 rows).
  -> Finished p4.CF.wt_pct: R2 = 0.038 | MAE = 11.155
  -> Skipping p4.WR_10kPa.wt_pct: Not enough data (0 rows).
  -> Skipping p4.WR_1500kPa.wt_pct: Not enough data (0 rows).
  -> Skipping p4.WR_33kPa.wt_pct: Not enough data (0 rows).

========== Evaluating Config: Spectral + pH + EC ==========
Features in use: 20 columns
  -> Finished p1.pH.index: R2 = 1.000 | MAE = 0.000
  -> Finished p1.EC.ds_m: R2 = 1.000 | MAE = 0.000
  -> Skipping p1.Clay.wt_pct: Not enough data (49 rows).
  -> Finished p1.Sand.wt_pct: R2 = 0.357 | MAE = 3.389
  -> Skipping p1.Silt.wt_pct: Not enough data (49 rows).
  -> Finished p2.N.wt_pct: R2 = 0.607 | MAE = 0.154
  -> Skipping p2.Zn.mg_kg: Not enough data (0 rows).
  -> Finished p2.OC.wt_pct: R2 = 0.556 | MAE = 3.291
  -> Skipping p3.Fe.mg_kg: 

c:\Users\jagda\anaconda3\Lib\site-packages\numpy\_core\_methods.py:53: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
